<a href="https://colab.research.google.com/github/SudheerkumarKuppala/Accident-Data-Analyzer/blob/main/Raining_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

In [7]:
!pip install flask xgboost joblib pyngrok

In [8]:
df = pd.read_csv("flood_risk_dataset_india 2.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (10000, 14)

Columns:
['Latitude', 'Longitude', 'Rainfall (mm)', 'Temperature (°C)', 'Humidity (%)', 'River Discharge (m³/s)', 'Water Level (m)', 'Elevation (m)', 'Land Cover', 'Soil Type', 'Population Density', 'Infrastructure', 'Historical Floods', 'Flood Occurred']

First 5 rows:


,Latitude,Longitude,Rainfall (mm),Temperature (°C),Humidity (%),River Discharge (m³/s),Water Level (m),Elevation (m),Land Cover,Soil Type,Population Density,Infrastructure,Historical Floods,Flood Occurred
0,18.861663,78.835584,218.999493,34.144337,43.912963,4236.182888,7.415552,377.465433,Water Body,Clay,7276.742184,1,0,1
1,35.570715,77.654451,55.353599,28.778774,27.585422,2472.585219,8.811019,7330.608875,Forest,Peat,6897.736956,0,1,0
2,29.227824,73.108463,103.991908,43.934956,30.108738,977.328053,4.631799,2205.873488,Agricultural,Loam,4361.518494,1,1,1
3,25.361096,85.610733,198.984191,21.569354,34.453690,3683.208933,2.891787,2512.277800,Desert,Sandy,6163.069701,1,1,0
4,12.524541,81.822101,144.626803,32.635692,36.292267,2093.390678,3.188466,2001.818223,Agricultural,Loam,6167.964591,1,0,0



Missing values:
Latitude                  0
Longitude                 0
Rainfall (mm)             0
Temperature (°C)          0
Humidity (%)              0
River Discharge (m³/s)    0
Water Level (m)           0
Elevation (m)             0
Land Cover                0
Soil Type                 0
Population Density        0
Infrastructure            0
Historical Floods         0
Flood Occurred            0
dtype: int64


In [9]:
# Remove duplicates
df = df.drop_duplicates()

# Remove missing values
df = df.dropna()

print("Cleaned dataset shape:", df.shape)

Cleaned dataset shape: (10000, 14)


In [10]:
categorical_cols = ["Land Cover", "Soil Type", "Infrastructure"]

for col in categorical_cols:
    print(f"\n{col} unique values:")
    print(df[col].unique())


Land Cover unique values:
['Water Body' 'Forest' 'Agricultural' 'Desert' 'Urban']

Soil Type unique values:
['Clay' 'Peat' 'Loam' 'Sandy' 'Silt']

Infrastructure unique values:
[1 0]


In [11]:
label_encoders = {}

categorical_cols = ["Land Cover", "Soil Type", "Infrastructure"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("Categorical columns encoded successfully.")

Categorical columns encoded successfully.


In [12]:
target_col = "Flood Occurred"

X = df.drop(columns=[target_col])
y = df[target_col]

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(target_col)

Feature columns:
['Latitude', 'Longitude', 'Rainfall (mm)', 'Temperature (°C)', 'Humidity (%)', 'River Discharge (m³/s)', 'Water Level (m)', 'Elevation (m)', 'Land Cover', 'Soil Type', 'Population Density', 'Infrastructure', 'Historical Floods']

Target column:
Flood Occurred


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (8000, 13)
X_test shape: (2000, 13)


In [14]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
}

best_model = None
best_accuracy = 0
best_model_name = ""

print("Model Accuracies:")
print("-" * 40)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print("-" * 40)
print("Best Model:", best_model_name)
print("Best Accuracy:", best_accuracy)

Model Accuracies:
----------------------------------------
Decision Tree: 0.4775
Random Forest: 0.5055
KNN: 0.5170


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:10:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost: 0.5050
----------------------------------------
Best Model: KNN
Best Accuracy: 0.517


In [15]:
best_pred = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, best_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50       966
           1       0.53      0.54      0.53      1034

    accuracy                           0.52      2000
   macro avg       0.52      0.52      0.52      2000
weighted avg       0.52      0.52      0.52      2000


Confusion Matrix:
[[479 487]
 [479 555]]


In [16]:
joblib.dump(best_model, "flood_model.pkl")
joblib.dump(X.columns.tolist(), "model_columns.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")

print("Saved files:")
print("- flood_model.pkl")
print("- model_columns.pkl")
print("- label_encoders.pkl")

Saved files:
- flood_model.pkl
- model_columns.pkl
- label_encoders.pkl


In [17]:
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("templates/ and static/ folders created")

templates/ and static/ folders created


In [18]:
app_code = r'''
from flask import Flask, render_template, request
import joblib
import pandas as pd

app = Flask(__name__)

# Load model and metadata
model = joblib.load("flood_model.pkl")
model_columns = joblib.load("model_columns.pkl")
label_encoders = joblib.load("label_encoders.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        input_data = {
            "Latitude": float(request.form["latitude"]),
            "Longitude": float(request.form["longitude"]),
            "Rainfall (mm)": float(request.form["rainfall"]),
            "Temperature (°C)": float(request.form["temperature"]),
            "Humidity (%)": float(request.form["humidity"]),
            "River Discharge (m³/s)": float(request.form["river_discharge"]),
            "Water Level (m)": float(request.form["water_level"]),
            "Elevation (m)": float(request.form["elevation"]),
            "Land Cover": request.form["land_cover"],
            "Soil Type": request.form["soil_type"],
            "Population Density": float(request.form["population_density"]),
            "Infrastructure": request.form["infrastructure"],
            "Historical Floods": int(request.form["historical_floods"])
        }

        # Encode categorical fields exactly like training
        for col in ["Land Cover", "Soil Type", "Infrastructure"]:
            le = label_encoders[col]
            value = str(input_data[col])

            if value not in le.classes_:
                return f"Error: Invalid value '{value}' for {col}. Allowed values: {list(le.classes_)}"

            input_data[col] = le.transform([value])[0]

        # Make DataFrame in exact training order
        input_df = pd.DataFrame([input_data])
        input_df = input_df[model_columns]

        prediction = model.predict(input_df)[0]

        if prediction == 1:
            result = "Flood Risk Detected"
            risk_class = "high"
        else:
            result = "No Flood Risk"
            risk_class = "safe"

        return render_template(
            "result.html",
            prediction_text=result,
            risk_class=risk_class
        )

    except Exception as e:
        return f"Error during prediction: {str(e)}"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully


In [19]:
index_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Rising Waters - Flood Prediction</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="container">
        <h1>Rising Waters</h1>
        <p class="subtitle">AI-Based Flood Risk Prediction System</p>

        <form action="/predict" method="post" class="prediction-form">
            <div class="grid">

                <div>
                    <label>Latitude</label>
                    <input type="number" step="any" name="latitude" required>
                </div>

                <div>
                    <label>Longitude</label>
                    <input type="number" step="any" name="longitude" required>
                </div>

                <div>
                    <label>Rainfall (mm)</label>
                    <input type="number" step="any" name="rainfall" required>
                </div>

                <div>
                    <label>Temperature (°C)</label>
                    <input type="number" step="any" name="temperature" required>
                </div>

                <div>
                    <label>Humidity (%)</label>
                    <input type="number" step="any" name="humidity" required>
                </div>

                <div>
                    <label>River Discharge (m³/s)</label>
                    <input type="number" step="any" name="river_discharge" required>
                </div>

                <div>
                    <label>Water Level (m)</label>
                    <input type="number" step="any" name="water_level" required>
                </div>

                <div>
                    <label>Elevation (m)</label>
                    <input type="number" step="any" name="elevation" required>
                </div>

                <div>
                    <label>Land Cover</label>
                    <select name="land_cover" required>
                        <option value="">Select</option>
                        <option value="Water Body">Water Body</option>
                        <option value="Forest">Forest</option>
                        <option value="Agricultural">Agricultural</option>
                        <option value="Desert">Desert</option>
                        <option value="Urban">Urban</option>
                    </select>
                </div>

                <div>
                    <label>Soil Type</label>
                    <select name="soil_type" required>
                        <option value="">Select</option>
                        <option value="Clay">Clay</option>
                        <option value="Peat">Peat</option>
                        <option value="Loam">Loam</option>
                        <option value="Sandy">Sandy</option>
                        <option value="Silt">Silt</option>
                    </select>
                </div>

                <div>
                    <label>Population Density</label>
                    <input type="number" step="any" name="population_density" required>
                </div>

                <div>
                    <label>Infrastructure</label>
                    <select name="infrastructure" required>
                        <option value="">Select</option>
                        <option value="0">0</option>
                        <option value="1">1</option>
                    </select>
                </div>

                <div>
                    <label>Historical Floods</label>
                    <select name="historical_floods" required>
                        <option value="">Select</option>
                        <option value="0">0 - No</option>
                        <option value="1">1 - Yes</option>
                    </select>
                </div>

            </div>

            <button type="submit">Predict Flood Risk</button>
        </form>
    </div>
</body>
</html>
'''

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(index_html)

print("templates/index.html created successfully")

templates/index.html created successfully


In [20]:
result_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prediction Result</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="container result-page">
        <h1>Prediction Result</h1>

        <div class="result-box {{ risk_class }}">
            <h2>{{ prediction_text }}</h2>
        </div>

        <a href="/" class="back-btn">Go Back</a>
    </div>
</body>
</html>
'''

with open("templates/result.html", "w", encoding="utf-8") as f:
    f.write(result_html)

print("templates/result.html created successfully")

templates/result.html created successfully


In [21]:
style_css = r'''
body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: linear-gradient(135deg, #dbeafe, #e0f2fe);
    color: #1f2937;
}

.container {
    max-width: 1100px;
    margin: 40px auto;
    background: #ffffff;
    padding: 30px;
    border-radius: 16px;
    box-shadow: 0 8px 30px rgba(0,0,0,0.12);
}

h1 {
    text-align: center;
    margin-bottom: 10px;
    color: #0f172a;
}

.subtitle {
    text-align: center;
    margin-bottom: 30px;
    color: #475569;
    font-size: 18px;
}

.prediction-form .grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 18px;
}

.prediction-form label {
    display: block;
    margin-bottom: 6px;
    font-weight: bold;
}

.prediction-form input,
.prediction-form select {
    width: 100%;
    padding: 10px;
    border: 1px solid #cbd5e1;
    border-radius: 10px;
    font-size: 15px;
    box-sizing: border-box;
}

.prediction-form button {
    display: block;
    width: 100%;
    margin-top: 25px;
    padding: 14px;
    background: #2563eb;
    color: white;
    font-size: 17px;
    border: none;
    border-radius: 10px;
    cursor: pointer;
}

.prediction-form button:hover {
    background: #1d4ed8;
}

.result-page {
    text-align: center;
}

.result-box {
    margin: 30px auto;
    padding: 30px;
    border-radius: 14px;
    font-size: 22px;
    font-weight: bold;
    width: 70%;
}

.result-box.high {
    background: #fee2e2;
    color: #b91c1c;
    border: 2px solid #ef4444;
}

.result-box.safe {
    background: #dcfce7;
    color: #166534;
    border: 2px solid #22c55e;
}

.back-btn {
    display: inline-block;
    margin-top: 20px;
    text-decoration: none;
    background: #334155;
    color: white;
    padding: 12px 20px;
    border-radius: 10px;
}

.back-btn:hover {
    background: #1e293b;
}

@media (max-width: 768px) {
    .prediction-form .grid {
        grid-template-columns: 1fr;
    }

    .result-box {
        width: 90%;
    }
}
'''

with open("static/style.css", "w", encoding="utf-8") as f:
    f.write(style_css)

print("static/style.css created successfully")

static/style.css created successfully


In [22]:
requirements_txt = '''
flask
pandas
scikit-learn
xgboost
joblib
pyngrok
'''

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_txt)

print("requirements.txt created successfully")

requirements.txt created successfully


In [25]:
!ngrok config add-authtoken "PASTE_YOUR_TOKEN_HERE"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [26]:
!pkill ngrok
!pkill -f app.py

In [28]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: on
Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [30]:
!pkill ngrok
!pkill -f app.py

In [31]:
import subprocess
import time

process = subprocess.Popen(["python", "app.py"])
time.sleep(8)

print("Flask app started in background")

Flask app started in background


In [32]:
!lsof -i:5000

COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 18091 root    5u  IPv4 446206      0t0  TCP *:5000 (LISTEN)
python3 18111 root    5u  IPv4 446206      0t0  TCP *:5000 (LISTEN)
python3 18111 root    6u  IPv4 446206      0t0  TCP *:5000 (LISTEN)


In [34]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: on
Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [35]:
!pkill ngrok
!pkill -f app.py

In [36]:
import subprocess
import time

process = subprocess.Popen(["python", "app.py"])
time.sleep(8)
print("Flask app started in background")

Flask app started in background


In [37]:
!lsof -i:5000

COMMAND   PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 19144 root    5u  IPv4 472386      0t0  TCP *:5000 (LISTEN)
python3 19164 root    5u  IPv4 472386      0t0  TCP *:5000 (LISTEN)
python3 19164 root    6u  IPv4 472386      0t0  TCP *:5000 (LISTEN)


In [39]:
!pkill -f app.py
!pkill ngrok || true

In [40]:
app_code = r'''
from flask import Flask, render_template, request
import joblib
import pandas as pd
import os

app = Flask(__name__)

# Load model and metadata
model = joblib.load("flood_model.pkl")
model_columns = joblib.load("model_columns.pkl")
label_encoders = joblib.load("label_encoders.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        input_data = {
            "Latitude": float(request.form["latitude"]),
            "Longitude": float(request.form["longitude"]),
            "Rainfall (mm)": float(request.form["rainfall"]),
            "Temperature (°C)": float(request.form["temperature"]),
            "Humidity (%)": float(request.form["humidity"]),
            "River Discharge (m³/s)": float(request.form["river_discharge"]),
            "Water Level (m)": float(request.form["water_level"]),
            "Elevation (m)": float(request.form["elevation"]),
            "Land Cover": request.form["land_cover"],
            "Soil Type": request.form["soil_type"],
            "Population Density": float(request.form["population_density"]),
            "Infrastructure": request.form["infrastructure"],
            "Historical Floods": int(request.form["historical_floods"])
        }

        # Encode categorical fields exactly like training
        for col in ["Land Cover", "Soil Type", "Infrastructure"]:
            le = label_encoders[col]
            value = str(input_data[col])

            if value not in le.classes_:
                return f"Error: Invalid value '{value}' for {col}. Allowed values: {list(le.classes_)}"

            input_data[col] = le.transform([value])[0]

        # Build DataFrame in exact training order
        input_df = pd.DataFrame([input_data])
        input_df = input_df[model_columns]

        prediction = model.predict(input_df)[0]

        if int(prediction) == 1:
            result = "Flood Risk Detected"
            risk_class = "high"
        else:
            result = "No Flood Risk"
            risk_class = "safe"

        return render_template("result.html",
                               prediction_text=result,
                               risk_class=risk_class)

    except Exception as e:
        return f"Error during prediction: {str(e)}"

if __name__ == "__main__":
    # IMPORTANT: host must be 0.0.0.0 for Colab public preview
    app.run(host="0.0.0.0", port=5000, debug=False)
'''
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py written")

app.py written


In [41]:
import os
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)
print("templates/ and static/ ready")

templates/ and static/ ready


In [42]:
index_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Rising Waters - Flood Prediction</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="container">
        <h1>Rising Waters</h1>
        <p class="subtitle">AI-Based Flood Risk Prediction System</p>

        <form action="/predict" method="post" class="prediction-form">
            <div class="grid">

                <div>
                    <label>Latitude</label>
                    <input type="number" step="any" name="latitude" required>
                </div>

                <div>
                    <label>Longitude</label>
                    <input type="number" step="any" name="longitude" required>
                </div>

                <div>
                    <label>Rainfall (mm)</label>
                    <input type="number" step="any" name="rainfall" required>
                </div>

                <div>
                    <label>Temperature (°C)</label>
                    <input type="number" step="any" name="temperature" required>
                </div>

                <div>
                    <label>Humidity (%)</label>
                    <input type="number" step="any" name="humidity" required>
                </div>

                <div>
                    <label>River Discharge (m³/s)</label>
                    <input type="number" step="any" name="river_discharge" required>
                </div>

                <div>
                    <label>Water Level (m)</label>
                    <input type="number" step="any" name="water_level" required>
                </div>

                <div>
                    <label>Elevation (m)</label>
                    <input type="number" step="any" name="elevation" required>
                </div>

                <div>
                    <label>Land Cover</label>
                    <select name="land_cover" required>
                        <option value="">Select</option>
                        <option value="Water Body">Water Body</option>
                        <option value="Forest">Forest</option>
                        <option value="Agricultural">Agricultural</option>
                        <option value="Desert">Desert</option>
                        <option value="Urban">Urban</option>
                    </select>
                </div>

                <div>
                    <label>Soil Type</label>
                    <select name="soil_type" required>
                        <option value="">Select</option>
                        <option value="Clay">Clay</option>
                        <option value="Peat">Peat</option>
                        <option value="Loam">Loam</option>
                        <option value="Sandy">Sandy</option>
                        <option value="Silt">Silt</option>
                    </select>
                </div>

                <div>
                    <label>Population Density</label>
                    <input type="number" step="any" name="population_density" required>
                </div>

                <div>
                    <label>Infrastructure</label>
                    <select name="infrastructure" required>
                        <option value="">Select</option>
                        <option value="0">0</option>
                        <option value="1">1</option>
                    </select>
                </div>

                <div>
                    <label>Historical Floods</label>
                    <select name="historical_floods" required>
                        <option value="">Select</option>
                        <option value="0">0 - No</option>
                        <option value="1">1 - Yes</option>
                    </select>
                </div>

            </div>

            <button type="submit">Predict Flood Risk</button>
        </form>
    </div>
</body>
</html>
'''
with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(index_html)

print("templates/index.html written")

templates/index.html written


In [43]:
result_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prediction Result</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="container result-page">
        <h1>Prediction Result</h1>

        <div class="result-box {{ risk_class }}">
            <h2>{{ prediction_text }}</h2>
        </div>

        <a href="/" class="back-btn">Go Back</a>
    </div>
</body>
</html>
'''
with open("templates/result.html", "w", encoding="utf-8") as f:
    f.write(result_html)

print("templates/result.html written")

templates/result.html written


In [44]:
style_css = r'''
body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: linear-gradient(135deg, #dbeafe, #e0f2fe);
    color: #1f2937;
}

.container {
    max-width: 1100px;
    margin: 40px auto;
    background: #ffffff;
    padding: 30px;
    border-radius: 16px;
    box-shadow: 0 8px 30px rgba(0,0,0,0.12);
}

h1 {
    text-align: center;
    margin-bottom: 10px;
    color: #0f172a;
}

.subtitle {
    text-align: center;
    margin-bottom: 30px;
    color: #475569;
    font-size: 18px;
}

.prediction-form .grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 18px;
}

.prediction-form label {
    display: block;
    margin-bottom: 6px;
    font-weight: bold;
}

.prediction-form input,
.prediction-form select {
    width: 100%;
    padding: 10px;
    border: 1px solid #cbd5e1;
    border-radius: 10px;
    font-size: 15px;
    box-sizing: border-box;
}

.prediction-form button {
    display: block;
    width: 100%;
    margin-top: 25px;
    padding: 14px;
    background: #2563eb;
    color: white;
    font-size: 17px;
    border: none;
    border-radius: 10px;
    cursor: pointer;
}

.prediction-form button:hover {
    background: #1d4ed8;
}

.result-page {
    text-align: center;
}

.result-box {
    margin: 30px auto;
    padding: 30px;
    border-radius: 14px;
    font-size: 22px;
    font-weight: bold;
    width: 70%;
}

.result-box.high {
    background: #fee2e2;
    color: #b91c1c;
    border: 2px solid #ef4444;
}

.result-box.safe {
    background: #dcfce7;
    color: #166534;
    border: 2px solid #22c55e;
}

.back-btn {
    display: inline-block;
    margin-top: 20px;
    text-decoration: none;
    background: #334155;
    color: white;
    padding: 12px 20px;
    border-radius: 10px;
}

.back-btn:hover {
    background: #1e293b;
}

@media (max-width: 768px) {
    .prediction-form .grid {
        grid-template-columns: 1fr;
    }

    .result-box {
        width: 90%;
    }
}
'''
with open("static/style.css", "w", encoding="utf-8") as f:
    f.write(style_css)

print("static/style.css written")

static/style.css written


In [45]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
^C


In [48]:
!pkill -f app.py

In [49]:
import subprocess
import time

process = subprocess.Popen(["python", "app.py"])
time.sleep(5)
print("Flask started in background")

Flask started in background


In [50]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)

https://5000-m-s-kkb-use4a2-tb3ir43nd58a-a.us-east4-2.prod.colab.dev


In [51]:
!pkill -f app.py

In [52]:
import subprocess, time

process = subprocess.Popen(
    ["python", "app.py"],
    stdout=open("flask.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)
print("Started Flask in background. PID:", process.pid)

Started Flask in background. PID: 24400


In [53]:
!ps -ef | grep app.py | grep -v grep

root       24400     578  9 14:38 ?        00:00:02 python3 app.py


In [54]:
!cat flask.log

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit


In [55]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)

https://5000-m-s-kkb-use4a2-tb3ir43nd58a-a.us-east4-2.prod.colab.dev


In [56]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(5000)")
print(url)

https://5000-m-s-kkb-use4a2-tb3ir43nd58a-a.us-east4-2.prod.colab.dev


In [57]:
!pkill -f app.py
!pkill ngrok || true

In [58]:
!pip install flask-ngrok pyngrok -q

In [59]:
app_code = r'''
from flask import Flask, render_template, request
import joblib
import pandas as pd
from flask_ngrok import run_with_ngrok

app = Flask(__name__)
run_with_ngrok(app)   # <-- important

# Load model and metadata
model = joblib.load("flood_model.pkl")
model_columns = joblib.load("model_columns.pkl")
label_encoders = joblib.load("label_encoders.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        input_data = {
            "Latitude": float(request.form["latitude"]),
            "Longitude": float(request.form["longitude"]),
            "Rainfall (mm)": float(request.form["rainfall"]),
            "Temperature (°C)": float(request.form["temperature"]),
            "Humidity (%)": float(request.form["humidity"]),
            "River Discharge (m³/s)": float(request.form["river_discharge"]),
            "Water Level (m)": float(request.form["water_level"]),
            "Elevation (m)": float(request.form["elevation"]),
            "Land Cover": request.form["land_cover"],
            "Soil Type": request.form["soil_type"],
            "Population Density": float(request.form["population_density"]),
            "Infrastructure": request.form["infrastructure"],
            "Historical Floods": int(request.form["historical_floods"])
        }

        for col in ["Land Cover", "Soil Type", "Infrastructure"]:
            le = label_encoders[col]
            value = str(input_data[col])
            if value not in le.classes_:
                return f"Invalid value for {col}: {value}. Allowed: {list(le.classes_)}"
            input_data[col] = le.transform([value])[0]

        input_df = pd.DataFrame([input_data])
        input_df = input_df[model_columns]

        prediction = model.predict(input_df)[0]

        if int(prediction) == 1:
            result = "Flood Risk Detected"
            risk_class = "high"
        else:
            result = "No Flood Risk"
            risk_class = "safe"

        return render_template("result.html", prediction_text=result, risk_class=risk_class)

    except Exception as e:
        return f"Error during prediction: {str(e)}"

if __name__ == "__main__":
    app.run()
'''
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py updated with flask-ngrok")

app.py updated with flask-ngrok


In [60]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.

Sign up for an account: https://dashboard.ngrok.com/signup
Get your credential: https://dashboard.ngrok.com/get-started/your-authtoken

ERR_NGROK_4018

Exception in thread Thread-1:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most r

In [61]:
import pandas as pd
import joblib

# load dataset and saved artifacts
df = pd.read_csv("flood_risk_dataset_india 2.csv")
model = joblib.load("flood_model.pkl")
model_columns = joblib.load("model_columns.pkl")
label_encoders = joblib.load("label_encoders.pkl")

# take one sample row
sample = df.iloc[[0]].copy()

# encode categorical columns if needed
for col in ["Land Cover", "Soil Type", "Infrastructure"]:
    if col in sample.columns:
        sample[col] = label_encoders[col].transform(sample[col].astype(str))

# keep exact training columns
sample = sample[model_columns]

pred = model.predict(sample)[0]
print("Prediction for first row:", pred)

Prediction for first row: 1


In [62]:
!zip -r rising_waters_project.zip app.py flood_model.pkl model_columns.pkl label_encoders.pkl templates static requirements.txt "flood_risk_dataset_india 2.csv"

  adding: app.py (deflated 63%)
  adding: flood_model.pkl (deflated 35%)
  adding: model_columns.pkl (deflated 14%)
  adding: label_encoders.pkl (deflated 54%)
  adding: templates/ (stored 0%)
  adding: templates/result.html (deflated 42%)
  adding: templates/index.html (deflated 80%)
  adding: static/ (stored 0%)
  adding: static/style.css (deflated 66%)
  adding: requirements.txt (stored 0%)
  adding: flood_risk_dataset_india 2.csv (deflated 54%)


In [63]:
from google.colab import files
files.download("rising_waters_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>